In [21]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np
import os
import pandas as pd
import json

In [ ]:
folder_path=r"VCdb"

In [ ]:
import os
import zipfile
import json
import pandas as pd

dataframes = {}

# Get all ZIP files and sort by filename
zip_files = sorted(
    [f for f in os.listdir(folder_path) if f.endswith('.zip')],
    reverse=True  # Latest name comes first
)

if not zip_files:
    print("No ZIP files found.")
else:
    latest_zip_name = zip_files[0]
    latest_zip_path = os.path.join(folder_path, latest_zip_name)
    print(f"Loading from latest ZIP file (by name): {latest_zip_name}")

    with zipfile.ZipFile(latest_zip_path, 'r') as z:
        for file_name in z.namelist():
            if file_name.endswith('.json'):
                try:
                    with z.open(file_name) as f:
                        data = json.load(f)

                    df = pd.json_normalize(data)
                    key = os.path.splitext(os.path.basename(file_name))[0]
                    dataframes[key] = df

                except json.JSONDecodeError as e:
                    print(f"JSON decoding failed for {file_name}: {e}")
                except Exception as e:
                    print(f"Failed to load {file_name}: {e}")




In [ ]:
df=(dataframes['BaseVehicle']
.merge(dataframes['Vehicle'],how='outer').merge(dataframes['Make'],how='outer')
.merge(dataframes['Model'],how='outer').merge(dataframes['VehicleType'],on='VehicleTypeID',how='outer')
.merge(dataframes['VehicleToEngineConfig'],on='VehicleID',how='outer')
.merge(dataframes['EngineConfig'],on='EngineConfigID',how='outer')
.merge(dataframes['Valves'],on='ValvesID',how='outer')
.merge(dataframes['Aspiration'],on='AspirationID',how='outer')
.merge(dataframes['EngineDesignation'],on='EngineDesignationID',how='outer')
.merge(dataframes['EngineVIN'],on='EngineVINID',how='outer')
.merge(dataframes['VehicleToDriveType'],on='VehicleID',how='outer').merge(dataframes['DriveType'],on='DriveTypeID',how='outer')
.merge(dataframes['EngineBase'],on='EngineBaseID',how='outer').merge(dataframes['FuelType'],on='FuelTypeID',how='outer')
.merge(dataframes['VehicleToBrakeConfig'],on='VehicleID',how='outer',suffixes=('_VehicleToBrakeConfig','_VehicleToBrakeConfig2'))
.merge(dataframes['BrakeConfig'],on='BrakeConfigID',how='outer').merge(dataframes['BrakeSystem'],on='BrakeSystemID',how='outer')
.merge(dataframes['Region'],on='RegionID',how='outer')
.merge(dataframes['VehicleToClass'],on='VehicleID',how='outer',suffixes=('_VehicleToClass','_VehicleToClass2')).merge(dataframes['Class'],on='ClassID',how='outer')
.merge(dataframes['VehicleToWheelbase'],on='VehicleID',how='outer',suffixes=('_VehicleToWheelbase','_VehicleToWheelbase2')).merge(dataframes['WheelBase'],left_on='WheelbaseID',right_on='WheelBaseID',how='outer')
.merge(dataframes['VehicleToBodyStyleConfig'],on='VehicleID',how='outer',suffixes=('VehicleToBodyStyleConfig','VehicleToBodyStyleConfig2')).merge(dataframes['BodyStyleConfig'],on='BodyStyleConfigID',how='outer').merge(dataframes['BodyType'],on='BodyTypeID',how='outer')
.merge(dataframes['EngineVersion'],on='EngineVersionID',how='outer')
)
df


In [ ]:
df_All=df[['BaseVehicleID',"VehicleTypeName", 'YearID','MakeName','ModelName','BlockType', 'Cylinders','Liter','ValvesPerEngine','CC','CID','FuelTypeName','AspirationName','EngineDesignationName','EngineVINName','DriveTypeName','RegionAbbr','BodyTypeName','WheelbaseID','WheelBase','WheelBaseMetric','EngineVersion','BrakeSystemName']] #"ClassName",
df_All = df_All.drop_duplicates()
df_All

In [26]:
datetext=latest_zip_name.split("_")[-1].split(".")[0]

In [ ]:
df_All.to_excel(fr"Output\{datetext}_Autocare_VCdb_MDHD.xlsx", index=False, engine='openpyxl',sheet_name='Autocare_VCdb')

In [ ]:
df = df[(df['MakeName'].str.contains("Thor Motor Coach", regex=False, na=False, case=False)) | (df['MakeName'].str.contains("Winnebago", regex=False, na=False, case=False))]
df = df[df['YearID'] > 2019]

df_filtered=df[['YearID','MakeName','ModelName','Liter', 'Cylinders','BlockType','FuelTypeName','BrakeSystemName',"VehicleTypeName"]] #"ClassName",
df_filtered = df_filtered.drop_duplicates()
df_filtered

In [ ]:
df.columns

In [ ]:
df.ModelName.unique()